In [18]:
from ireal_transformer import load_ireal_transformer
from parser import Tune
import torch.nn.functional as F

In [2]:
data = open("data/data.txt", encoding="utf-8").read().strip()

tunes = Tune.parse_ireal_url(data, verbose=False)

In [3]:
model = load_ireal_transformer("ireal_transformer_model.pth")

/home/ottenl/projects/jazz-transformer/ireal_transformer.py:100: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=layers)


In [6]:
embeddings = [model.get_song_embedding(tune, causal=False) for tune in tunes]

In [ ]:
print(f"Parsed {len(tunes)} tunes.")
print(f"First tune: {tunes[0].title}")
print(f"Embedding shape: {embeddings[0].shape}")
print(embeddings[0])

Parsed 1955 tunes.
First tune: 9.20 Special
Embedding shape: torch.Size([128])
tensor([-1.2449e+00,  1.2527e+00,  1.5732e-01,  2.6571e+00, -2.3590e+00,
         9.6173e-01,  4.1191e-01, -8.8740e-02,  2.9953e-01, -6.8846e-01,
         3.3871e-01, -1.4079e+00, -1.1977e+00,  1.3352e+00, -1.1488e+00,
        -5.3462e-01, -2.3563e+00,  3.1525e-01,  2.9905e-04, -2.1925e-01,
        -2.4713e+00, -3.5051e+00, -1.1520e+00, -1.9960e+00,  4.8781e-01,
         1.0580e+00, -4.8988e-01,  1.5459e+00, -1.0549e+00,  1.3681e+00,
        -1.6064e+00, -2.6800e+00, -1.6880e+00, -6.3456e-01,  5.2941e-01,
        -1.4242e+00, -1.1791e+00, -1.2052e+00,  3.0195e-01,  6.2330e-01,
        -1.2075e+00,  2.1132e-01,  7.8869e-01, -6.4127e-01, -1.5500e+00,
         7.8109e-02, -9.2503e-01,  6.5201e-01, -3.1092e-01, -7.6426e-01,
        -1.0815e+00,  7.2431e-01,  1.6632e+00, -3.0957e-01, -6.8663e-01,
        -4.7677e-01,  3.4851e-01, -4.4369e-01,  1.5384e+00, -2.0224e+00,
        -4.5186e-01, -8.7269e-01, -6.2002e-02

In [10]:
import pandas as pd
df = pd.DataFrame()
for i, tune in enumerate(tunes):
    df = pd.concat([df, pd.DataFrame({
        "title": [tune.title],
        "embedding": [embeddings[i].numpy()]
    })], ignore_index=True)

title                                        How High The Moon
embedding    [-1.36297, -0.744271, 1.501025, 2.700562, -0.5...
Name: 858, dtype: object

In [28]:
song = df[df["title"] == "Lush Life"].iloc[0]

song_embedding = torch.as_tensor(song["embedding"], dtype=torch.float32)
embeddings_tensor = torch.stack(embeddings)

df[f"similarity_to_{song['title']}"] = F.cosine_similarity(
    embeddings_tensor,
    song_embedding.unsqueeze(0),
    dim=1
).cpu().numpy()

df.sort_values(
    by=f"similarity_to_{song['title']}",
    ascending=False
).head(20)

,title,embedding,distance_to_How High The Moon,similarity_to_How High The Moon,similarity_to_C-Jam Blues,similarity_to_Giant Steps,similarity_to_Donna Lee,similarity_to_Countdown,similarity_to_Lush Life
1283,Lush Life,"[-0.138773, -0.495188, 0.891928, 2.693182, -2....",0.727189,0.745973,0.447481,0.844680,0.674859,0.877852,1.000000
1655,Quiet Now,"[-0.775161, -0.029143, 1.658975, 3.331521, -1....",0.727189,0.751302,0.361032,0.807015,0.715524,0.832092,0.936538
1864,Spring Can Really Hang You Up The Most,"[-1.320107, -0.007848, 1.284848, 3.360571, -1....",0.727189,0.823162,0.442804,0.830951,0.776952,0.840412,0.925506
594,Ecaroh,"[0.322241, -0.58229, 0.890174, 3.198682, -1.95...",0.727189,0.799698,0.529622,0.861167,0.735899,0.882116,0.917254
1639,Prelude To A Kiss,"[-1.073868, 0.093304, 1.552503, 3.156444, -1.4...",0.727189,0.746129,0.380373,0.767497,0.737877,0.787600,0.902079
402,Chelsea Bridge,"[-1.003342, -0.204763, 1.375003, 2.725475, -1....",0.727189,0.859606,0.570788,0.811901,0.848753,0.806227,0.900510
159,Ask Me Now,"[-1.200446, -0.052645, 1.591323, 3.640405, -1....",0.727189,0.817104,0.538817,0.789077,0.768176,0.792649,0.899153
1580,Pannonica,"[-0.204513, -0.437194, 2.164376, 2.751609, -1....",0.727189,0.817964,0.554679,0.849428,0.722251,0.855127,0.897737
582,"Duke, The","[-1.302387, 0.393966, 1.192312, 2.753241, -2.6...",0.727189,0.734018,0.453302,0.779090,0.666244,0.780939,0.893678
31,A Sleepin' Bee,"[-0.855225, -0.097295, 1.55842, 2.811645, -1.3...",0.727189,0.835709,0.531238,0.815548,0.800563,0.840412,0.891069


In [31]:
import song_similarity
song_similarity.save_song_embeddings(model, tunes)